# ME-HSI data walkthrough: open, store, view, summarise

An onboarding notebook for new team members. It covers the **data layer** of `spectral-select` end to end, before any deep learning. Every code cell is preceded by a short story: what the cell does, why you will need it, the science behind it, and why the project settled on this particular way of doing it.

1. What a multi-excitation hyperspectral (ME-HSI) dataset *is*, and the names we use for its parts
2. Where datasets live on disk and how they are organised
3. Where the metadata comes from (exposure times, lamp power, reference frames)
4. Four ways to open a single 3D cube from a raw `.im3` file
5. Opening a full 4D dataset from a processed pickle, and reading the pickle schema directly
6. Building a dataset by hand and storing it (pickle, npy, TIFF, HDF5)
7. Viewing slices, band montages and false-colour composites
8. Spectra at a pixel, over a region, across excitations, and as an excitation-emission matrix
9. Summary statistics, histograms, saturation checks, per-class spectra
10. Preprocessing with the functional API (the same operations the GUI wizard applies)
11. A five-minute hand-off to the band-selection algorithm and to SpectraForge

**Before you start**

* Install with `pip install -e ".[all]"` following `docs/onboarding/SETUP.md`. Reading `.im3` files needs a JDK and Maven; that guide covers it.
* Put data under `Data/` as described in `docs/onboarding/DATA_GUIDE.md`, or point `SPECTRAL_SELECT_DATA` at another `Data` folder.
* Launch from the repository: `jupyter lab examples/03_hsi_data_walkthrough.ipynb`.

Every section that needs a file is guarded. Without any data the notebook synthesises a small cube with SpectraForge, so every cell still runs. The whole notebook takes two to four minutes on a laptop.

### Setup: finding the repository and the data

**Why this cell exists.** Every path in this repository is expressed relative to the repository root, and the experiment scripts assume you run them from there. Notebooks, however, start their kernel in whichever folder they live in (`examples/`). Rather than hard-coding `..`, the cell walks up the directory tree until it finds `pyproject.toml`; that makes the notebook work from `examples/`, from the root, or from a copy somewhere else.

**Why a `DATA_ROOT` override.** `Data/` is git-ignored and can be several gigabytes, so people keep it on an external disk or a shared mount. `SPECTRAL_SELECT_DATA` lets you point at it without editing the notebook.

**Why `Lichens_2`.** It is the smallest real dataset we have (50 MB of raw cubes, 256 x 348 pixels, 8 excitations) and it comes with everything: raw `.im3` files, both metadata spreadsheets, processed pickles, an analysis mask and class annotations. Every example in the onboarding material uses it so the numbers you see match the guides.

**Why a temporary output directory.** The notebook writes pickles, caches, TIFFs and a model. Writing them into `Data/` or `results/` would pollute folders that scripts read; a `tempfile` directory keeps the repository clean and is printed so you can inspect it.

In [ ]:
from __future__ import annotations

import json
import os
import pickle
import re
import sys
import tempfile
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["figure.dpi"] = 90


def find_project_root(start: Path = Path.cwd()) -> Path:
    # Walk up until pyproject.toml is found, so the notebook works from examples/ or the repo root.
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the spectral-select repository")


PROJECT_ROOT = find_project_root()
DATA_ROOT = Path(os.environ.get("SPECTRAL_SELECT_DATA", PROJECT_ROOT / "Data"))
RAW_DIR, PROC_DIR = DATA_ROOT / "Raw", DATA_ROOT / "processed"

# The sample used throughout. Lichens_2 is the smallest real dataset (50 MB raw, 256 x 348 pixels).
SAMPLE = "Lichens_2"
RAW_SAMPLE_DIR = RAW_DIR / SAMPLE
PROC_SAMPLE_DIR = PROC_DIR / SAMPLE
PROCESSED_PKL = PROC_SAMPLE_DIR / "data_dual_cutoff_40nm.pkl"                  # raw counts, dual cutoff applied
NORMALISED_PKL = PROC_SAMPLE_DIR / "data_cutoff_40nm_exposure_max_power_min.pkl"  # same, exposure + power normalised

HAVE_RAW = RAW_SAMPLE_DIR.is_dir() and any(RAW_SAMPLE_DIR.glob("*.im3"))
HAVE_PROCESSED = PROCESSED_PKL.exists()

# Scratch directory for everything this notebook writes.
OUT_DIR = Path(tempfile.mkdtemp(prefix="hsi_walkthrough_"))

import spectral_select
from spectral_select import ExcitationData, SpectraData

print(f"spectral_select {spectral_select.__version__}  |  python {sys.version.split()[0]}")
print(f"project root : {PROJECT_ROOT}")
print(f"data root    : {DATA_ROOT}   (raw .im3 sample: {HAVE_RAW}, processed pickle: {HAVE_PROCESSED})")
print(f"scratch dir  : {OUT_DIR}")

## 1. The data model

### The physics in two paragraphs

Fluorescence is the emission of light by a molecule shortly after it absorbs a photon. The emitted photon carries less energy than the absorbed one, so emission always sits at a **longer wavelength than excitation** (the Stokes shift). Each fluorescent molecule (a *fluorophore*: collagen, elastin, NADH, FAD, chlorophyll, ...) has its own excitation spectrum (which wavelengths it absorbs) and emission spectrum (which wavelengths it emits). A biological sample is a mixture of fluorophores, so the emission spectrum you record depends on which excitation you used: a 310 nm photon lights up collagen and tryptophan, a 365 nm photon lights up NADH, a 450 nm photon lights up FAD.

That is the whole reason for **multi-excitation** hyperspectral imaging. One excitation gives one emission spectrum per pixel and cannot separate fluorophores whose emission overlaps. Stepping the excitation gives a two-dimensional fingerprint per pixel, the **excitation-emission matrix (EEM)**, and the differences between excitations are exactly where the material contrast hides. Our instrument couples a **tunable light source** (TLS, a xenon lamp behind a monochromator) that steps the excitation through 310, 325, 340, 365, 385, 400, 415, 430 nm with a **Nuance multispectral camera** (a liquid-crystal tunable filter in front of a CCD) that records the emission from 420 to 720 nm in 10 nm steps, one exposure per band. One excitation therefore produces a 3D **cube** of 31 emission bands, and the set of cubes is the **4D dataset**.

```
4D dataset  =  { excitation_nm  ->  cube[height, width, n_emission_bands] }

               310 nm       325 nm       340 nm     ...     430 nm
            +----------+ +----------+ +----------+       +----------+
  H rows    |  H x W   | |  H x W   | |  H x W   |  ...  |  H x W   |    each plane = one emission band
            |  x 31    | |  x 31    | |  x 31    |       |  x 31    |
            +----------+ +----------+ +----------+       +----------+
```

### Why the band count differs per excitation

After preprocessing the band count differs per excitation (22 to 30 in the Lichens data), because two instrumental artefacts are cut out: bands close to the excitation line (scattered excitation light, not fluorescence) and bands around twice the excitation wavelength (a grating harmonic). Section 10 shows both. The spatial size is always shared across excitations, which is what lets the autoencoder look at all excitations of a pixel at once.

### Vocabulary

| Term | Meaning in this repository |
|---|---|
| **cube** | `np.ndarray` of shape `(H, W, n_bands)` for one excitation. Always height, width, bands, in that order. |
| **excitation** (`ex`, `ex_nm`) | Illumination wavelength in nm. Dictionary key of the dataset, stored as a `float` such as `365.0`. |
| **emission band**, **band** | One plane of a cube. `emission_wavelengths[i]` is its centre wavelength in nm. |
| **slice** | A 2D image: one excitation, one band. |
| **spectrum** | The 1D vector along the band axis at one pixel, or averaged over a region. |
| **EEM** | Excitation-emission matrix: intensity over (excitation, emission) for one pixel or region. |
| **mask** | `(H, W)` array, 1 = pixel used in the analysis, 0 = ignored. Class masks hold integer class ids instead. |
| `ExcitationData` | The Python object for one cube plus its wavelengths, exposure time and lamp power. |
| `SpectraData` | The 4D container: `{ex_nm: ExcitationData}` plus mask, sample name and metadata. |
| **Rayleigh cutoff** | Bands below `ex + offset` are removed (scattered excitation light, not fluorescence). |
| **second-order cutoff** | Bands inside `2*ex +/- offset` are removed (grating second-order artefact). |

**Why `(H, W, bands)` and not `(bands, H, W)`.** Image libraries (ImageJ, tifffile) prefer the band axis first; we put it last because every operation in this project is "per pixel, along the spectrum": `cube[row, col]` is a spectrum, `cube[mask]` is a matrix of spectra, and the autoencoder's channel axis is the last one. The convention is enforced by `ExcitationData` (it validates `len(emission_wavelengths) == cube.shape[2]`), so an accidentally transposed cube fails fast instead of producing nonsense.

## 2. Where the data lives

**Why look at the folder before opening anything.** The folder layout *is* the contract. The loader finds cubes by globbing `*.im3` in the top level of a sample folder, reads the excitation wavelength from the file name, and looks for two spreadsheets in fixed places. If a new acquisition does not follow this layout, nothing downstream works, so the first thing you do with a new dataset is compare its folder against this picture.

`Data/` is git-ignored: datasets are gigabytes and travel through the lab's shared storage (or Zenodo for the public Lichens release), never through git. Raw and processed data sit side by side, and **raw folders are treated as read-only**: the only files ever added to them are the two metadata spreadsheets. The full catalogue of datasets is in `docs/onboarding/DATA_GUIDE.md`.

```
Data/
├── Raw/<Sample>/                       # straight from the instrument
│   ├── 310.im3, 325.im3, ...           # one Nuance cube per excitation, named by excitation wavelength
│   ├── metadata.xlsx                   # exposure time per excitation        (columns: Excitation | Exposure)
│   ├── TLS Scans/average_power.xlsx    # lamp power per excitation           (Excitation Wavelength (nm) | Average Power (W))
│   ├── Reflectance/White.im3           # optional white reference            (Lichens_2)
│   └── Background.im3, Whitelight.im3  # optional dark / white-light frames  (Drop Data)
└── processed/<Sample>/                 # what the loader or the GUI wrote
    ├── *.pkl                           # 4D datasets (two dialects, see section 5)
    ├── *_mask.npy, *_class_mask.npy, class_mask.png    # analysis and class masks
    └── roi_regions.json, *_class_info.json             # annotation metadata
```

The cell below prints the two folders with file sizes. Notice that each raw cube is about 5.8 MB (348 x 256 pixels x 31 bands x 2 bytes) while a processed pickle is hundreds of megabytes: the processed files hold every excitation, as float64, sometimes twice (section 5 explains why).

In [ ]:
def tree(root: Path, max_entries: int = 25) -> None:
    # Print a directory with file sizes in MB.
    if not root.exists():
        print(f"[missing] {root}")
        return
    print(root)
    entries = sorted(root.rglob("*"))
    for p in entries[:max_entries]:
        size = f"{p.stat().st_size / 1e6:8.1f} MB" if p.is_file() else "      <dir>"
        print(f"  {size}  {p.relative_to(root)}")
    if len(entries) > max_entries:
        print(f"  ... {len(entries) - max_entries} more")


tree(RAW_SAMPLE_DIR)
print()
tree(PROC_SAMPLE_DIR)

## 3. Metadata: exposure, lamp power, reference frames

### Why two spreadsheets exist at all

A camera does not measure fluorescence; it counts photons over an exposure. The number of counts in a pixel is roughly

```
counts  ∝  (lamp power at λex)  x  (exposure time)  x  (fluorescence yield of the pixel)  x  (detector efficiency at λem)
```

The first two factors change with every excitation, and by a lot. The xenon lamp behind the monochromator is weak in the ultraviolet and strong in the blue, so the operator compensates with long exposures at 310 nm (5.4 seconds in Lichens_2) and short ones at 400 nm (12 ms). If you compared raw counts across excitations you would mostly be comparing lamp brightness. Dividing by exposure and by lamp power turns counts into something proportional to the fluorescence yield, which is the quantity that carries chemistry. That is **radiometric normalisation**, and it needs the two numbers per excitation that these spreadsheets hold:

* `metadata.xlsx`: exposure time per excitation, in **milliseconds** as exported by the camera software.
* `TLS Scans/average_power.xlsx`: average lamp power per excitation, computed from the power-meter scans (`.TRQ` files) taken with the tunable light source. Units are whatever the meter exported; check the magnitude before comparing datasets (Lichens is in W, Drop Data was recorded in mW).

**Why the Drop Data lesson matters.** The first Drop Data analysis ignored these factors because the exposures were encoded in the file names rather than in a spreadsheet. Selections looked plausible and were wrong: the autoencoder had learned lamp brightness. The radiometric rerun (exposure x lamp power correction) fixed the results. Always attach both numbers to every `ExcitationData` (`exposure_time`, `laser_power`) before training anything.

The cell merges the two sheets into one table; the `_val` helper pulls a single number out of a one-row slice and turns missing values into `None`, which is what the normalisation functions expect for "unknown, skip this excitation".

In [ ]:
def read_acquisition_metadata(raw_dir: Path) -> pd.DataFrame:
    # Merge metadata.xlsx (exposure) and TLS Scans/average_power.xlsx (power) into one table.
    table = pd.read_excel(raw_dir / "metadata.xlsx")
    table.columns = table.columns.str.strip()
    table = table.rename(columns={"Excitation": "excitation_nm", "Exposure": "exposure_ms"})
    power_path = raw_dir / "TLS Scans" / "average_power.xlsx"
    if power_path.exists():
        power = pd.read_excel(power_path).iloc[:, :2]
        power.columns = ["excitation_nm", "power_W"]
        table = table.merge(power, on="excitation_nm", how="left")
    table["excitation_nm"] = table["excitation_nm"].astype(float)
    return table.sort_values("excitation_nm").reset_index(drop=True)


def _val(row, col):
    # Float from a one-row DataFrame slice, or None if the column is absent or NaN.
    if row is None or len(row) == 0 or col not in row:
        return None
    v = float(row[col].iloc[0])
    return None if np.isnan(v) else v


if HAVE_RAW and (RAW_SAMPLE_DIR / "metadata.xlsx").exists():
    meta = read_acquisition_metadata(RAW_SAMPLE_DIR)
    display(meta)
else:
    meta = None
    print("no raw sample folder: skipping the metadata spreadsheets")

### Two file-name conventions

Most datasets name each cube by its excitation (`365.im3`), which is what the project loader parses. The Drop Data acquisition did something smarter and, for the loader, unfortunate: it recorded **two or three exposure brackets per excitation** (`365 10 SPF.im3`, `365 12 SPF.im3`, `365 15 SPF.im3`) plus a dark frame (`Background.im3`) and a white-light frame (`Whitelight.im3`). Bracketing is the standard answer to a detector with limited dynamic range: the bright drops saturate at a long exposure while the faint substrate is lost at a short one, so you record both and merge them (high dynamic range imaging). The radiometric rerun does exactly that.

The loader parses the excitation with `float(name.split(".")[0])`, so any of those names raises `ValueError` and aborts the load. Until the loader learns the second convention, Drop Data cubes are read with PyImageJ directly (section 4b) and the parsing logic below is what the archived Drop Data scripts use. Keep it in mind when you name your own acquisitions: **bare numbers for sample cubes, and reference frames in a subfolder**.

In [ ]:
# Two filename conventions exist under Data/Raw. The project loader understands only the first one.
#   "365.im3"          -> excitation 365 nm                           (Lichens, Collagen, Sponges)
#   "365 10 SPF.im3"   -> excitation 365 nm, 10 ms exposure bracket   (Drop Data, plus Background.im3 / Whitelight.im3)
SIMPLE_RE = re.compile(r"^(?P<ex>\d+(?:\.\d+)?)$")
BRACKET_RE = re.compile(r"^(?P<ex>\d+)\s+(?P<exposure_ms>\d+)\s+SPF$", re.IGNORECASE)


def parse_im3_name(path: Path) -> dict:
    stem = path.stem
    if m := SIMPLE_RE.match(stem):
        return {"role": "sample", "excitation_nm": float(m["ex"]), "exposure_ms": None}
    if m := BRACKET_RE.match(stem):
        return {"role": "sample", "excitation_nm": float(m["ex"]), "exposure_ms": float(m["exposure_ms"])}
    if stem.lower() in {"background", "dark"}:
        return {"role": "background", "excitation_nm": None, "exposure_ms": None}
    if stem.lower() in {"whitelight", "white"}:
        return {"role": "white_reference", "excitation_nm": None, "exposure_ms": None}
    return {"role": "unknown", "excitation_nm": None, "exposure_ms": None}


for name in ["365.im3", "310 1500 SPF.im3", "Background.im3", "Whitelight.im3", "notes.im3"]:
    print(f"{name:20s} -> {parse_im3_name(Path(name))}")

## 4. Opening a single 3D cube from `.im3`

### What an `.im3` is, and why we go through Java to read it

`.im3` is the PerkinElmer/CRi **Nuance** multispectral format: a proprietary container with a small header (`FileVersion`, `ProgramName: Nuance`) followed by `uint16` planes. There is no public specification. Two options existed when the project started: reverse-engineer the container and write a NumPy reader, or use the reader that already exists in **Bio-Formats**, the open-source library behind Fiji/ImageJ that understands about 150 microscopy formats and has a maintained `IM3Reader`. We chose Bio-Formats: a home-made parser risks silently misreading a header change, whereas Bio-Formats is tested against files from many instruments and gives us every other microscopy format for free. The price is a Java virtual machine, reached from Python through **PyImageJ** (which uses JPype to embed the JVM). That is the only reason the setup guide asks for a JDK and Maven.

### Rules that save hours

1. **Start ImageJ once per Python process.** JPype can start one JVM per process and cannot restart it; a second `imagej.init()` returns a broken gateway (it reports version `Inactive`). Keep one `ij` object and hand it around; the helper below does that with a module-level cache.
2. The first `imagej.init("sc.fiji:fiji")` asks Maven to resolve Fiji and downloads about 350 MB into `~/.jgo` and `~/.m2`. Later starts take a few seconds. `mode="headless"` avoids starting a GUI event loop, which would fight with Jupyter and Qt.
3. Two lines of console noise are normal in headless mode and can be ignored: `WARNING: package sun.awt.X11 not in java.desktop` and `[ERROR] Cannot create plugin: ...JavaScriptScriptLanguage`.
4. The file exposes no wavelength table through Bio-Formats (we checked: zero global-metadata keys, no channel wavelengths), so the emission axis is **reconstructed from the acquisition protocol**: 420 nm + 10 nm x band index, except that for excitations above 400 nm the first band is `ex + 20` nm, because the filter cannot measure below the excitation and the protocol shifted the start. That rule lives in `HyperspectralDataLoader.load_data`. If the instrument settings ever change, that is the line to update, and section 10's cutoff plot is how you would notice.

In [ ]:
_IJ = None


def get_imagej(endpoint: str = "sc.fiji:fiji"):
    # Start the ImageJ/Fiji gateway once and reuse it. Pass a local install path
    # such as "/Applications/Fiji" to skip the Maven download.
    global _IJ
    if _IJ is None:
        import imagej  # pip install pyimagej  (needs a JDK + Maven, see docs/onboarding/SETUP.md)
        t0 = time.time()
        _IJ = imagej.init(endpoint, mode="headless")
        print(f"ImageJ {_IJ.getVersion()} ready in {time.time() - t0:.1f}s")
    return _IJ


try:
    ij = get_imagej()
    HAVE_IMAGEJ = True
except Exception as exc:  # ImportError, or a JVM / Maven problem
    ij = None
    HAVE_IMAGEJ = False
    print("PyImageJ is not available in this environment:", repr(exc)[:300])
    print("-> the .im3 cells below are skipped; see docs/onboarding/SETUP.md, section 'Reading .im3 files'")

### 4a. Way A: the project loader, `HyperspectralDataLoader`

**Why start with the loader.** This class is the historical heart of the project (it predates everything else) and it is what the GUI wizard and `SpectraData.from_raw` call underneath. It does four things in one call: reads every `<excitation>.im3` in a folder, attaches exposure times from `metadata.xlsx`, reconstructs the emission axis, and optionally applies the Rayleigh and second-order cutoffs. Understanding its output (`raw_data` = uncut, `data` = cut) explains the shape of every processed pickle in `Data/processed`.

**Why `use_fiji=False` and then inject the gateway.** The loader wants to start its own ImageJ in its constructor. We already started one (rule 1 above), so we construct it with `use_fiji=False` and hand it our gateway. This is a small hack, but it is the reason this notebook can show the loader and the direct route side by side in one kernel.

**Why `cutoff_offset=40`.** The Lichens paper runs use 40 nm. Smaller offsets (20, 30) leave a scatter tail in the ultraviolet excitations where the tunable filter's passband is broad; 60 nm removes real emission at the long excitations (the 430 nm cube would lose everything below 490 nm). Forty keeps at least 22 bands per cube and removes the artefacts at every excitation. The processed file names under `Data/processed/Lichens_2` carry the number so nobody has to guess.

The summary table shows the effect: 31 raw bands everywhere, 22 to 30 after the cutoffs, and the exposure attached from the spreadsheet.

In [ ]:
if HAVE_RAW and HAVE_IMAGEJ:
    from mehsi_preprocessor.io.hyperspectral_loader import HyperspectralDataLoader

    loader = HyperspectralDataLoader(
        data_path=str(RAW_SAMPLE_DIR),
        metadata_path=str(RAW_SAMPLE_DIR / "metadata.xlsx"),
        cutoff_offset=40,        # nm; the Lichens paper runs used 40
        use_fiji=False,          # do not start a second JVM ...
        verbose=False,
    )
    loader._ij, loader.use_fiji = ij, True   # ... reuse the running gateway instead

    t0 = time.time()
    loader.load_data(apply_cutoff=True)      # fills loader.raw_data (uncut) and loader.data (cut)
    print(f"loaded {len(loader.excitation_wavelengths)} excitations in {time.time() - t0:.1f}s: "
          f"{loader.excitation_wavelengths}")

    rows = []
    for ex in loader.excitation_wavelengths:
        raw_cube, raw_wl = loader.get_cube(ex, processed=False)
        cut_cube, cut_wl = loader.get_cube(ex, processed=True)
        rows.append({"excitation_nm": ex, "raw_shape": raw_cube.shape, "raw_em_range": (raw_wl[0], raw_wl[-1]),
                     "bands_after_cutoff": cut_cube.shape[2], "cut_em_range": (cut_wl[0], cut_wl[-1]),
                     "exposure_ms": loader.raw_data[str(ex)]["expos_val"]})
    display(pd.DataFrame(rows))
else:
    loader = None
    print("skipped (needs the raw sample folder and PyImageJ)")

**Why look at the cutoff before anything else.** The loader's `visualize_cutoff` plots the mean spectrum of one excitation before and after the two cutoffs. For 365 nm you should see: the full spectrum (blue) starting at 420 nm, a shaded Rayleigh region up to 405 nm (empty here, because 420 is already above it), and a shaded second-order window around 730 nm that removes the last bands (690 to 720 nm). If the green curve ever cuts through a real emission peak, the offset is too large for that excitation; if a spike survives near the excitation or near twice the excitation, it is too small. This one plot is the quickest sanity check for a new acquisition.

In [ ]:
if loader is not None:
    fig = loader.visualize_cutoff(365.0)   # mean spectrum before and after the dual cutoff
    plt.show()

### 4b. Way B: under the hood, with PyImageJ directly

**When you need this.** The loader is convenient but rigid: bare-number file names, all files in a folder, float64 output. The direct route is what you use when the convention differs (Drop Data brackets, reference frames), when you want the native `uint16` counts to check saturation, or when you want a single file quickly. It is also what the loader does internally, so seeing it demystifies the loader: `ij.io().open()` returns a Java image, `ij.py.from_java()` converts it to an `xarray.DataArray` with dims `(row, col, ch)`, and `.values` is the NumPy array.

**Why `float32` here.** Raw counts fit in `uint16`, but every downstream operation (cutoffs are a slice, but normalisation divides, the autoencoder trains in float32) wants floats. `float32` has 24 bits of mantissa, far more than the 12 bits of the camera, and halves memory compared with the loader's `float64`. The processed pickles in `Data/processed` are `float64` for historical reasons only.

**Why `nuance_emission_axis` is a separate function.** It encodes the acquisition protocol (section 4, rule 4) in one place, mirrors the loader exactly, and is the function you would change if the camera were reconfigured.

In [ ]:
def nuance_emission_axis(n_bands: int, excitation_nm: float, step_nm: float = 10.0) -> list[float]:
    # Emission band centres (nm) for one Nuance cube, reconstructed from the acquisition protocol.
    # Mirrors HyperspectralDataLoader: the scan starts at 420 nm, or at ex + 20 nm for ex > 400 nm.
    start = 420.0 if excitation_nm <= 400.0 else excitation_nm + 20.0
    return [start + i * step_nm for i in range(n_bands)]


def read_im3(path: Path, ij) -> np.ndarray:
    # Read one .im3 cube as a float32 (H, W, n_bands) array through Fiji / Bio-Formats.
    img = ij.io().open(str(path))
    xarr = ij.py.from_java(img)                    # xarray.DataArray, dims ('row', 'col', 'ch')
    if tuple(xarr.dims) != ("row", "col", "ch"):   # be defensive about axis order
        xarr = xarr.transpose("row", "col", "ch")
    return np.asarray(xarr.values, dtype=np.float32)


if HAVE_RAW and HAVE_IMAGEJ:
    im3_path = RAW_SAMPLE_DIR / "365.im3"
    xarr = ij.py.from_java(ij.io().open(str(im3_path)))
    print("xarray dims :", xarr.dims, "shape", xarr.shape, "dtype", xarr.dtype, "(native counts are uint16)")
    cube_365 = read_im3(im3_path, ij)
    em_365 = nuance_emission_axis(cube_365.shape[2], 365.0)
    print("numpy cube  :", cube_365.shape, cube_365.dtype, f"counts {cube_365.min():.0f}..{cube_365.max():.0f}")
    print("emission nm :", em_365[:4], "...", em_365[-2:])
else:
    cube_365 = None

### 4c. Way C: cache cubes as `.npy` and reopen instantly

**Why cache.** Starting the JVM and decoding `.im3` costs seconds per file, needs Java on the machine, and cannot run in some places (a cluster node, CI). NumPy's `.npy` format reopens in milliseconds, needs nothing but NumPy, and can be **memory-mapped**: `np.load(..., mmap_mode="r")` maps the file and reads only the pages you touch, so a script that wants one band never loads the other thirty. `Data/processed/Drop Data/raw/*.npy` is exactly this kind of cache and is why the Drop Data scripts run without ImageJ.

**Why a JSON sidecar.** A bare array forgets which excitation it is and what its band axis means. Writing `{"excitation_nm": ..., "emission_nm": [...]}` next to it costs nothing and prevents the classic mistake of pairing a cube with the wrong wavelengths. Keep the excitation in the file name too, so a directory listing is self-describing.

In [ ]:
CACHE_DIR = OUT_DIR / "npy_cache"
CACHE_DIR.mkdir(exist_ok=True)

if cube_365 is not None:
    np.save(CACHE_DIR / "365.npy", cube_365)
    (CACHE_DIR / "365.json").write_text(json.dumps({"excitation_nm": 365.0, "emission_nm": em_365}))

    t0 = time.time()
    cached = np.load(CACHE_DIR / "365.npy", mmap_mode="r")   # lazy: pages are read on access
    print(f"reopened {cached.shape} from cache in {1e3 * (time.time() - t0):.1f} ms; "
          f"identical: {np.array_equal(cached, cube_365)}")

### 4d. Way D: the one-liner, `SpectraData.from_raw`

In a script (a fresh Python process, so ImageJ has not been started yet) the whole folder becomes a 4D object in one call:

```python
from spectral_select import SpectraData, LoadingOptions

data = SpectraData.from_raw(
    "Data/Raw/Lichens_2",
    metadata_path="Data/Raw/Lichens_2/metadata.xlsx",
    loading_options=LoadingOptions(cutoff_offset=40),
)
```

It wraps way A (`DataLoader` -> `HyperspectralDataLoader`) and builds the `ExcitationData` objects for you. It starts its own ImageJ gateway internally, which is why it is not executed here after section 4a. Only `cutoff_offset` and the cutoff on/off flags of `LoadingOptions` are honoured on this path; the normalisation flags are documentation, and normalisation is a separate step (section 10).

## 5. Opening the 4D dataset from a processed pickle

### Why pickle, of all formats

A 4D dataset is *ragged*: each excitation has a different number of bands after the cutoffs, so it does not fit a single array. It is naturally a dictionary of arrays plus a few scalars. Python's pickle stores that structure exactly, with NumPy arrays serialised at full speed and no schema to maintain, and it is what the autoencoder pipeline consumed from day one. The trade-offs are known and accepted: pickles are Python-only, and **they execute code when loaded**, so only open files produced by this pipeline or handed to you by a colleague, never a pickle from an untrusted source. For sharing with people outside Python we export TIFF or HDF5 (section 6).

### The two dialects you will meet

* **Loader dialect** (`data_*.pkl`, written by `HyperspectralDataLoader.save_to_pkl` and `HyperspectralProcessor`): keys `data`, `raw_data`, `metadata`, `excitation_wavelengths`, `cutoff_offset`. Each `data[ex]` also carries the uncut cube under `raw`, so the file is roughly twice the size of the data. This is the older format and it exists because the early pipeline wanted to be able to redo the cutoff without going back to Java.
* **Export dialect** (`spectra_masked.pkl`, `spectra_unmasked.pkl`, `spectra_data.pkl`, written by `SpectraData.to_pickle`, the GUI and SpectraForge): keys `data`, `excitation_wavelengths`, plus optional `mask`, `exposure_times`, `laser_powers`. This is the canonical interchange format today: compact, and it carries the mask and the radiometric metadata that the loader dialect lacks.

`SpectraData.from_pickle` reads both by dispatching on the keys. Excitation keys are **strings** in the file (`"365.0"`) and **floats** in the object (`365.0`). `from_pickle` names the sample after the file stem, not after anything stored inside, and it keeps every unknown top-level key in `data.metadata` (for the loader dialect that includes the whole `raw_data`, which is how section 10 gets the uncut cubes without Java).

The summary helper below is worth keeping in your own toolbox: one row per excitation with shape, wavelength range, value range and the attached metadata is the fastest way to see whether a file is what you think it is.

In [ ]:
def summarize(data: SpectraData) -> pd.DataFrame:
    rows = []
    for ex in data.excitation_wavelengths:
        e = data.get_excitation(ex)
        rows.append({"excitation_nm": ex, "shape (H,W,bands)": e.shape, "dtype": e.cube.dtype.name,
                     "em_min": e.emission_wavelengths[0], "em_max": e.emission_wavelengths[-1],
                     "min": float(np.nanmin(e.cube)), "max": float(np.nanmax(e.cube)),
                     "exposure": e.exposure_time, "power": e.laser_power})
    return pd.DataFrame(rows)


if HAVE_PROCESSED:
    t0 = time.time()
    data = SpectraData.from_pickle(PROCESSED_PKL)
    print(f"loaded {PROCESSED_PKL.name} ({PROCESSED_PKL.stat().st_size / 1e6:.0f} MB) in {time.time() - t0:.1f}s")
    print(f"sample={data.sample_name!r}  spatial={data.spatial_shape}  excitations={data.n_excitations}  "
          f"mask={'yes' if data.mask is not None else 'no'}  extra metadata keys={list(data.metadata)}")
    display(summarize(data))
else:
    data = None
    print("processed pickle not found; a fallback dataset is built two cells below")

**Why open the raw dictionary as well.** `SpectraData` is a convenience wrapper; the file on disk is a plain nested dictionary, and sooner or later you will meet a pickle that the wrapper does not understand (an old experiment, a colleague's script). The `describe` helper prints the structure of any nested dict without printing the arrays themselves, which is the safe way to look inside a multi-hundred-megabyte object. Compare its output with the two dialects above: you should recognise `data`, `raw_data`, `metadata`, and inside `raw_data` the record fields the loader wrote in section 4a (`ex`, `em_start`, `expos_val`, `data`, `em_arr`).

In [ ]:
def describe(obj, name: str = "root", depth: int = 0, max_depth: int = 3, max_keys: int = 6) -> None:
    # Print the structure of a nested dict / array without printing the arrays themselves.
    pad = "  " * depth
    if isinstance(obj, dict):
        keys = list(obj)
        print(f"{pad}{name}: dict[{len(keys)}] {keys[:max_keys]}{' ...' if len(keys) > max_keys else ''}")
        if depth < max_depth:
            for k in keys[:max_keys]:
                describe(obj[k], str(k), depth + 1, max_depth, max_keys)
    elif isinstance(obj, np.ndarray):
        print(f"{pad}{name}: ndarray{obj.shape} {obj.dtype}")
    elif isinstance(obj, (list, tuple)):
        print(f"{pad}{name}: {type(obj).__name__}[{len(obj)}] {obj[:4]}{' ...' if len(obj) > 4 else ''}")
    else:
        print(f"{pad}{name}: {type(obj).__name__} = {str(obj)[:60]}")


if HAVE_PROCESSED:
    with open(PROCESSED_PKL, "rb") as fh:
        raw_pkl = pickle.load(fh)          # the file as Python sees it, before SpectraData wraps it
    describe(raw_pkl, max_keys=3)
else:
    raw_pkl = {}

### The working dataset and its mask

**Why a fallback.** New team members often set up the code before they have the data. If no processed pickle exists, the cell builds the dataset from what the loader read in section 4a; if there is no raw data either, it synthesises a small cube with SpectraForge (section 11). Either way, every later cell has a `data` object to work on, and the notebook doubles as an environment smoke test.

**Why a mask.** The camera sees the whole field of view: the specimen, but also the substrate, the ruler, the edges of the slide. Those pixels would dominate any statistic (they are most of the image) and would teach the autoencoder about the background rather than the sample. A binary **analysis mask** (1 = analyse, 0 = ignore) restricts everything that follows to the specimen. For Lichens_2 it lives next to the pickle as `lichens_2_mask.npy` and keeps about a quarter of the pixels; the export dialect carries it inside the file; when nothing exists we fall back to all ones so the code path is the same.

In [ ]:
if data is None and loader is not None:
    # Build the 4D object from what the loader read in section 4a.
    data = SpectraData(
        excitations={ex: ExcitationData(excitation_nm=ex,
                                        cube=loader.data[str(ex)]["cube"],
                                        emission_wavelengths=list(loader.data[str(ex)]["wavelengths"]),
                                        exposure_time=loader.raw_data[str(ex)]["expos_val"])
                     for ex in loader.excitation_wavelengths},
        sample_name=SAMPLE)
    print("built SpectraData from the raw loader")
elif data is None:
    # No data at all: synthesise a small 3-excitation cube with SpectraForge (section 11 explains it).
    from spectraforge.demo import build_demo
    data, _forge_gt = build_demo()
    print("built a synthetic SpectraData with SpectraForge")

# A binary analysis mask: from the file, from the sidecar .npy, or all ones.
mask_path = PROC_SAMPLE_DIR / "lichens_2_mask.npy"
if data.mask is not None:
    mask = data.mask > 0
elif mask_path.exists() and np.load(mask_path).shape == data.spatial_shape:
    mask = np.load(mask_path).astype(bool)
else:
    mask = np.ones(data.spatial_shape, dtype=bool)

ex0 = data.excitation_wavelengths[0]
print(f"working dataset: {data.sample_name}, {data.n_excitations} excitations, {data.spatial_shape}, "
      f"mask keeps {mask.mean():.0%} of the pixels")

## 6. Building and storing a dataset yourself

**Why you will do this.** Data arrives in many shapes: cubes from PyImageJ, TIFF stacks from a collaborator, arrays from a simulation, a subset you cropped for an experiment. The way to make any of them usable by the rest of the repository is the same: wrap each `(H, W, bands)` array in an `ExcitationData` with its wavelengths and its acquisition metadata, collect them in a `SpectraData`, and write the export dialect with `to_pickle`. From then on the GUI, `Analyzer` and every experiment driver can read it.

**Why attach exposure and power now.** They come from the spreadsheets of section 3 and they are the inputs of the radiometric normalisation in section 10. A dataset without them cannot be normalised later without going back to the raw folder, so we attach them at construction time. The `_val` helper returns `None` for anything missing, which the normalisation functions interpret as "skip".

**Why check the round trip.** `to_pickle` and `from_pickle` are the backbone of every hand-off between tools; the cell proves that cubes survive bit-for-bit, and it also shows the two things that do *not* survive by design: the sample name (recovered from the file stem) and free-form metadata. If you need them, put them in the file name or in a sidecar JSON.

In [ ]:
excitations = {}
for ex in data.excitation_wavelengths:
    e = data.get_excitation(ex)
    row = meta[meta.excitation_nm == ex] if meta is not None else None
    excitations[ex] = ExcitationData(
        excitation_nm=ex,
        cube=e.cube.astype(np.float32),                     # float32 halves the file size; plenty for camera counts
        emission_wavelengths=list(e.emission_wavelengths),
        exposure_time=_val(row, "exposure_ms") if row is not None else e.exposure_time,
        laser_power=_val(row, "power_W") if row is not None else e.laser_power,
    )

mine = SpectraData(
    excitations=excitations,
    mask=mask.astype(np.uint8),
    sample_name=f"{data.sample_name}_walkthrough",
    metadata={"source": str(PROCESSED_PKL if HAVE_PROCESSED else "synthetic"), "note": "built in 03_hsi_data_walkthrough"},
)

pkl_path = mine.to_pickle(OUT_DIR / "walkthrough_spectra.pkl")
back = SpectraData.from_pickle(pkl_path)
print(f"wrote {pkl_path.name}: {pkl_path.stat().st_size / 1e6:.1f} MB")
print("round trip identical:", np.array_equal(back.get_excitation(ex0).cube, mine.get_excitation(ex0).cube),
      "| exposure kept:", back.get_excitation(ex0).exposure_time,
      "| sample_name kept:", back.sample_name == mine.sample_name, "(from_pickle uses the file stem)",
      "| metadata kept:", bool(back.metadata), "(to_pickle drops it)")

### Storage formats we use, and when

Each format below answers a different question: *who* needs to read the data, and *how much* of it at a time.

| Format | Written by | Read by | Use it for |
|---|---|---|---|
| `.pkl` export dialect | `SpectraData.to_pickle`, GUI step 8, SpectraForge export | `SpectraData.from_pickle`, `Analyzer`, GUI step 1, experiment drivers | **The canonical interchange format.** One file per dataset variant. |
| `.pkl` loader dialect | `HyperspectralDataLoader.save_to_pkl`, `HyperspectralProcessor` | `SpectraData.from_pickle` (extra keys land in `.metadata`) | Legacy. Keeps the uncut cubes, so it is large. |
| `.npy` per cube | `np.save` | `np.load(..., mmap_mode="r")` | Fast caches of raw counts (Drop Data). Keep a JSON sidecar with the axis. |
| `.npz` + `.json` | `spectraforge.GroundTruth.save` | `np.load` | Synthetic ground truth (concentration maps, clean cubes). |
| multi-page `.tif` | `tifffile`, `experiments/export_tiffs.py`, `Analyzer.save_results` | Fiji / ImageJ, `tifffile.imread` | Sharing slices with people who work in ImageJ. Axis order is `(bands, H, W)`. |
| `.h5` | optional, `h5py` | `h5py` | Large datasets with partial reads. Not used by the pipeline today. |
| `.png` / `.npy` masks | GUI step 8, `ROIWidget.save_mask` | `load_ground_truth_from_png`, `np.load` | Analysis masks and class annotations. |

**Why TIFF for collaborators.** Microscopists live in Fiji. A multi-page TIFF with ImageJ metadata opens there as a stack with a slider over the bands, and writing the wavelength into each slice label means the person on the other side sees "em 520 nm" instead of "slice 11". Note the axis order: ImageJ wants `(bands, H, W)`, the opposite of our convention, hence the `np.moveaxis`.

**Why HDF5 exists in this table although nothing uses it.** Pickle loads everything or nothing. When a dataset no longer fits in memory (the full-resolution Lichens Dataset 1 is 1.5 GB per pickle), HDF5 with **per-band chunks** lets a script read one slice from disk without touching the rest. The cell shows the layout we would adopt: one group per excitation, the cube chunked `(64, 64, 1)`, the wavelengths stored beside it. It is a ten-line migration if it is ever needed.

In [ ]:
import h5py
import tifffile

e = data.get_excitation(ex0)

# Multi-page TIFF: ImageJ expects (bands, H, W). Slice labels carry the wavelengths into Fiji.
tif_path = OUT_DIR / f"{data.sample_name}_ex{ex0:.0f}.tif"
tifffile.imwrite(tif_path, np.moveaxis(e.cube.astype(np.float32), -1, 0), imagej=True,
                 metadata={"axes": "CYX", "Labels": [f"em {w:.0f} nm" for w in e.emission_wavelengths]})
stack = tifffile.imread(tif_path)
print(f"TIFF stack {tif_path.name}: {stack.shape}  (open it in Fiji with File > Open)")

# HDF5: one group per excitation, chunked per band so a single slice is a cheap partial read.
h5_path = OUT_DIR / f"{data.sample_name}.h5"
H, W = data.spatial_shape
with h5py.File(h5_path, "w") as h5:
    h5.attrs["sample_name"] = data.sample_name
    h5.create_dataset("mask", data=mask.astype(np.uint8), compression="gzip")
    for ex in data.excitation_wavelengths:
        g = h5.create_group(f"ex_{ex:.0f}")
        g.create_dataset("cube", data=data.get_excitation(ex).cube.astype(np.float32),
                         chunks=(min(64, H), min(64, W), 1), compression="gzip")
        g.create_dataset("emission_nm", data=np.asarray(data.get_excitation(ex).emission_wavelengths))
with h5py.File(h5_path) as h5:
    one_slice = h5[f"ex_{ex0:.0f}/cube"][:, :, 0]           # reads only that band from disk
    print(f"HDF5 {h5_path.name}: groups {list(h5)[:4]} ..., one slice read -> {one_slice.shape}")

## 7. Viewing: slices, band montages, false colour

### Why a helper that converts wavelengths to indices

A slice is `cube[:, :, band]`. The temptation is to write `cube[:, :, 10]`, and it is wrong more often than not: after the cutoffs, band 10 is 520 nm in the 310 nm cube and 560 nm in the 430 nm cube. Always ask for a wavelength and let `band_index` find the nearest band, so the same code shows the same emission under every excitation.

### Why a percentile stretch instead of min-max

Fluorescence images are heavy-tailed: a few bright specks (a saturated grain, a reflective edge) sit far above the tissue. A linear map from the minimum to the maximum would push the tissue into the darkest few percent of the colour map and you would see a black image with white dots. Clipping the colour scale at the 99th percentile shows the structure; the clipped pixels are the ones you inspect separately in section 9 (saturation). The `magma` colour map is perceptually uniform and prints well in greyscale, which matters for papers.

In [ ]:
def band_index(e: ExcitationData, em_nm: float) -> int:
    # Index of the band whose centre is closest to em_nm.
    return int(np.argmin(np.abs(np.asarray(e.emission_wavelengths) - em_nm)))


def show_slice(data: SpectraData, ex: float, em_nm: float, ax=None, percentile: float = 99.0, cmap: str = "magma"):
    e = data.get_excitation(ex)
    i = band_index(e, em_nm)
    img = e.cube[:, :, i]
    ax = plt.gca() if ax is None else ax
    im = ax.imshow(img, cmap=cmap, vmin=0, vmax=np.nanpercentile(img, percentile))
    ax.set_title(f"ex {ex:.0f} nm / em {e.emission_wavelengths[i]:.0f} nm", fontsize=9)
    ax.axis("off")
    return im


fig, ax = plt.subplots(figsize=(6, 4.5))
im = show_slice(data, ex0, 520, ax=ax)
plt.colorbar(im, ax=ax, label="intensity")
plt.show()

**Why a montage across excitations at one emission band.** This is the first picture that shows why we bother with multiple excitations. Every panel is the same emission wavelength (about 500 nm) and the same pixels; only the illumination differs. Regions that are bright at 310 or 340 nm and dark at 400 nm contain fluorophores that absorb in the ultraviolet (collagen, NADH); regions that light up only at longer excitations contain something else (flavins, chlorophyll in the lichens' algal partner). If all panels looked identical, multi-excitation imaging would be pointless; they do not, and that difference is what the band-selection method is trying to keep.

Each panel has its own colour scale (its own 99th percentile), because the raw counts are not comparable across excitations until section 10 normalises them.

In [ ]:
exs = data.excitation_wavelengths
ncols = int(np.ceil(len(exs) / 2))
fig, axes = plt.subplots(2, ncols, figsize=(3.2 * ncols, 5.5))
axes = np.atleast_1d(axes).ravel()
for ax, ex in zip(axes, exs):
    show_slice(data, ex, 500, ax=ax)
for ax in axes[len(exs):]:
    ax.axis("off")
fig.suptitle("The same emission band (about 500 nm) under every excitation", y=1.0)
plt.tight_layout()
plt.show()

**Why an emission sweep within one excitation, on a shared colour scale.** The previous montage varied the excitation; this one fixes the excitation and steps through its emission bands with a single colour scale. It shows the shape of the emission spectrum as a movie: where in the image the signal lives at 420 nm, where it has moved by 600 nm, and how quickly it fades. Bands that look like copies of their neighbours are redundant, which is the intuition behind the whole project: of the roughly 200 (excitation, emission) bands in a dataset, a handful carry the contrast and the rest repeat it. The desktop viewer (`launch_viewer()`) animates exactly this sweep.

In [ ]:
e = data.get_excitation(ex0)
idx = list(range(0, e.n_bands, max(1, e.n_bands // 8)))[:8]
fig, axes = plt.subplots(1, len(idx), figsize=(2.3 * len(idx), 2.6))
vmax = np.nanpercentile(e.cube, 99.5)
for ax, i in zip(np.atleast_1d(axes), idx):
    ax.imshow(e.cube[:, :, i], cmap="magma", vmin=0, vmax=vmax)
    ax.axis("off")
    ax.set_title(f"{e.emission_wavelengths[i]:.0f} nm", fontsize=9)
fig.suptitle(f"Emission sweep at excitation {ex0:.0f} nm (shared colour scale)")
plt.tight_layout()
plt.show()

### False-colour RGB

**Why false colour.** The eye integrates three broad bands; a hyperspectral cube has twenty to thirty narrow ones. To *see* spectral differences you have to map three of them to red, green and blue. Which three is a choice:

* **Chosen wavelengths** (`compose_false_color`): pick bands where you expect the chemistry to differ, for example a long, a middle and a short emission (620 / 540 / 470 nm). Each channel is stretched independently, so a pixel's hue encodes the *shape* of its spectrum rather than its brightness. This is the composite you put in a paper when you want the reader to see a specific band.
* **Automatic picks** (`create_rgb_image`): bands at 20 / 50 / 80 % of the band axis; a quick look when you have no hypothesis.
* **PCA false colour**: the 22 to 31 bands are strongly correlated (neighbouring wavelengths carry nearly the same image). Principal component analysis finds the three orthogonal directions of largest variance across pixels and maps them to RGB. It usually separates materials better than any three raw bands, precisely because it uses all of them; the cost is that the channels no longer mean a wavelength. PCA is fitted on masked pixels only so that the background does not define the axes. The same redundancy is what makes band selection possible: if three components explain most of the variance, a few well-chosen bands can too.
* **Max projection**: the brightest value along the spectrum, a quick "where is any signal" map.

A NumPy warning about `matmul` can appear on Apple Silicon inside PCA; it comes from the Accelerate BLAS library and is harmless, so the cell silences it.

In [ ]:
from sklearn.decomposition import PCA
from spectral_select.viewer import compose_false_color, create_rgb_image

e = data.get_excitation(ex0)
r, g, b = (band_index(e, nm) for nm in (620, 540, 470))
rgb_manual = compose_false_color(e.cube, r_band=r, g_band=g, b_band=b, percentile=99)
rgb_auto = create_rgb_image(e.cube, method="rgb", percentile=99)


def pca_false_color(cube: np.ndarray, mask: np.ndarray | None = None, percentile: float = 99.0) -> np.ndarray:
    H, W, B = cube.shape
    X = np.nan_to_num(cube.reshape(-1, B))
    fit_on = X[mask.reshape(-1)] if mask is not None else X
    # Accelerate BLAS on macOS + NumPy 2 raises spurious "overflow / invalid value encountered in matmul" here;
    # the decomposition itself is correct (float64, values < 1e4), so the warning is silenced.
    warnings.filterwarnings("ignore", message=".*encountered in matmul", category=RuntimeWarning)
    scores = PCA(n_components=3).fit(fit_on).transform(X).reshape(H, W, 3)
    out = np.empty_like(scores)
    for k in range(3):
        lo, hi = np.percentile(scores[..., k], [100 - percentile, percentile])
        out[..., k] = np.clip((scores[..., k] - lo) / (hi - lo + 1e-12), 0, 1)
    return out


fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
axes[0].imshow(rgb_manual)
axes[0].set_title(f"R/G/B = {e.emission_wavelengths[r]:.0f}/{e.emission_wavelengths[g]:.0f}/{e.emission_wavelengths[b]:.0f} nm")
axes[1].imshow(rgb_auto)
axes[1].set_title("create_rgb_image (20/50/80 % bands)")
axes[2].imshow(pca_false_color(e.cube, mask))
axes[2].set_title("PCA false colour (PC1-3)")
axes[3].imshow(np.nanmax(e.cube, axis=2), cmap="gray")
axes[3].set_title("max projection over bands")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

**Why an interactive slider is optional.** Exploring a cube is much faster with a slider than with static plots, and `ipywidgets` gives one in three lines. It is optional because it needs a running kernel (it does nothing in a rendered notebook on GitHub) and an extra package, so the static plots above stay the reference. Outside Jupyter there are two viewers: the desktop **ME-HSI Viewer** (`from spectral_select import launch_viewer; launch_viewer()`, a Tk app that opens `.pkl` files or raw folders, with band animation, click-to-plot spectra and histograms) and the **preprocessing wizard** (`spectral-select-gui`). Both are described in `docs/onboarding/components/`.

In [ ]:
# Optional interactive slider (needs ipywidgets). The static plots above cover the same ground.
try:
    from ipywidgets import Dropdown, IntSlider, interact

    def _view(ex=ex0, band=0):
        e = data.get_excitation(ex)
        band = min(band, e.n_bands - 1)
        plt.figure(figsize=(6, 4.5))
        plt.imshow(e.cube[:, :, band], cmap="magma", vmin=0, vmax=np.nanpercentile(e.cube[:, :, band], 99))
        plt.title(f"ex {ex:.0f} nm / em {e.emission_wavelengths[band]:.0f} nm")
        plt.axis("off")
        plt.show()

    interact(_view, ex=Dropdown(options=data.excitation_wavelengths, value=ex0),
             band=IntSlider(0, 0, max(x.n_bands for x in data.excitations.values()) - 1))
except ImportError:
    print("ipywidgets not installed (pip install ipywidgets)")

## 8. Spectra: pixel, region, all excitations, EEM

**Why spectra, not images, are the unit of analysis.** Every algorithm in this repository (the autoencoder, KNN classification, clustering) treats a pixel as a vector of intensities along the spectral axis; images are just how those vectors are arranged. Looking at spectra directly is therefore the most honest view of the data, and the first place to spot problems: a flat-topped curve means saturation, a spike at the excitation wavelength means the Rayleigh cutoff was too small, a smooth hump is fluorescence.

**Why a pixel under every excitation.** The overlay below is the 4D structure in one plot: one curve per excitation for the same pixel. The curves differ in shape, not only in height; that is the excitation-dependence that a single-excitation instrument cannot see. We pick the median masked pixel so that the choice is reproducible and safely inside the specimen; `extract_multi_excitation_spectrum` is the library helper that does the gathering.

In [ ]:
from spectral_select.viewer import extract_multi_excitation_spectrum

ys, xs = np.nonzero(mask)
row, col = int(np.median(ys)), int(np.median(xs))       # a pixel safely inside the mask
spectra = extract_multi_excitation_spectrum(data, x=col, y=row)   # {excitation_nm: spectrum}

fig, ax = plt.subplots(figsize=(8, 4))
for ex, spec in spectra.items():
    ax.plot(data.get_excitation(ex).emission_wavelengths, spec, marker=".", label=f"ex {ex:.0f} nm")
ax.set(xlabel="emission (nm)", ylabel="intensity", title=f"pixel (row {row}, col {col}) under every excitation")
ax.legend(ncol=2, fontsize=8)
plt.show()

**Why average over a region, with a spread.** A single pixel is noisy: photon shot noise scales with the square root of the counts, and at the short exposures of the blue excitations a pixel may hold only a few hundred counts. Averaging over a region of interest (ROI) of a few hundred pixels shrinks that noise by an order of magnitude, and the standard deviation band tells you whether the region is homogeneous (narrow band) or a mixture (wide band). Class-mean spectra like these are what the KNN classifier separates and what the papers plot. The `roi_mean_spectrum` helper uses boolean indexing `cube[roi_mask]`, which returns an `(n_pixels, bands)` matrix; that idiom is worth remembering, it is how every per-region computation in the repository works.

In [ ]:
def roi_mean_spectrum(e: ExcitationData, roi_mask: np.ndarray):
    pix = e.cube[roi_mask]                        # (n_pixels, bands)
    return np.nanmean(pix, axis=0), np.nanstd(pix, axis=0)


# A 21 x 21 window around the pixel, intersected with the analysis mask.
roi = np.zeros_like(mask)
roi[max(row - 10, 0):row + 11, max(col - 10, 0):col + 11] = True
roi &= mask

fig, ax = plt.subplots(figsize=(8, 4))
for ex in data.excitation_wavelengths:
    e = data.get_excitation(ex)
    m, s = roi_mean_spectrum(e, roi)
    (line,) = ax.plot(e.emission_wavelengths, m, label=f"ex {ex:.0f} nm")
    ax.fill_between(e.emission_wavelengths, m - s, m + s, color=line.get_color(), alpha=0.15)
ax.set(xlabel="emission (nm)", ylabel="intensity", title=f"mean +/- std over {int(roi.sum())} pixels")
ax.legend(ncol=2, fontsize=8)
plt.show()

**Why the excitation-emission matrix is the signature plot of this field.** Fold the overlay into a 2D image with excitation on one axis and emission on the other and you get the EEM: the fluorescence fingerprint of a material, the same object that fluorescence spectroscopists tabulate for pure compounds (collagen peaks near ex 330 / em 390 nm, NADH near 340 / 460, FAD near 450 / 535). Reading it: a bright blob is a fluorophore; blobs sit above the diagonal because emission is always redder than excitation (Stokes shift); the empty wedge at the lower left is the Rayleigh cutoff and the gaps along the anti-diagonal are the second-order cutoff. Comparing the EEMs of two regions is the quickest way to guess whether they differ chemically before running any model. The matrix is built on the union of all emission axes so that the cut bands appear as `NaN` (white) instead of being silently interpolated.

In [ ]:
def eem_matrix(data: SpectraData, roi_mask: np.ndarray):
    # Excitation-emission matrix of the ROI mean. Bands removed by the cutoffs stay NaN.
    em_axis = sorted({w for e in data.excitations.values() for w in e.emission_wavelengths})
    M = np.full((data.n_excitations, len(em_axis)), np.nan)
    for i, ex in enumerate(data.excitation_wavelengths):
        e = data.get_excitation(ex)
        m, _ = roi_mean_spectrum(e, roi_mask)
        for w, v in zip(e.emission_wavelengths, m):
            M[i, em_axis.index(w)] = v
    return M, em_axis


M, em_axis = eem_matrix(data, roi)
fig, ax = plt.subplots(figsize=(9, 3.8))
im = ax.imshow(M, aspect="auto", cmap="viridis", origin="lower",
               extent=[em_axis[0] - 5, em_axis[-1] + 5, -0.5, data.n_excitations - 0.5])
ax.set_yticks(range(data.n_excitations))
ax.set_yticklabels([f"{ex:.0f}" for ex in data.excitation_wavelengths])
ax.set(xlabel="emission (nm)", ylabel="excitation (nm)", title="excitation-emission matrix of the ROI (gaps = cutoff bands)")
plt.colorbar(im, ax=ax, label="mean intensity")
plt.show()

## 9. Statistics: per band, per excitation, masked against unmasked, per class

**Why these three checks on every new dataset.**

1. **Dynamic range per excitation.** If the mean is a few dozen counts, the exposure was too short (or the lamp too weak) and the band is mostly noise; if the 99th percentile touches the ceiling, it was too long. Both are acquisition problems you want to catch before spending GPU hours.
2. **Saturation.** The Nuance detector clips at **3886 counts** in our data (the maximum ever observed in raw cubes; the archived Drop Data scripts use the same constant). A saturated pixel has a flat-topped spectrum that looks like a very bright, featureless material; a handful of them can dominate a variance-based statistic and mislead the autoencoder. The check only makes sense on raw counts; after normalisation the ceiling is no longer a fixed number, so use the range check instead.
3. **Masked against unmasked.** If a statistic changes a lot when you apply the mask, the background was driving it. That is the argument for masking everything, including training.

The per-band table is written to CSV so that a dataset's quality report can be attached to a lab notebook entry. Long-format tables (one row per excitation and band) are deliberately chosen over wide ones: pandas can group, filter and plot them without reshaping.

In [ ]:
SATURATION_COUNTS = 3886   # ceiling observed in raw Nuance cubes; meaningless after normalisation


def band_statistics(data: SpectraData, mask: np.ndarray | None = None) -> pd.DataFrame:
    rows = []
    for ex in data.excitation_wavelengths:
        e = data.get_excitation(ex)
        pix = e.cube[mask] if mask is not None else e.cube.reshape(-1, e.n_bands)
        for i, w in enumerate(e.emission_wavelengths):
            v = pix[:, i]
            rows.append({"excitation_nm": ex, "emission_nm": w, "mean": np.nanmean(v), "std": np.nanstd(v),
                         "p99": np.nanpercentile(v, 99), "max": np.nanmax(v),
                         "saturated_frac": float(np.mean(v >= SATURATION_COUNTS))})
    return pd.DataFrame(rows)


stats = band_statistics(data, mask)
per_excitation = (stats.groupby("excitation_nm")
                  .agg(bands=("emission_nm", "size"), mean=("mean", "mean"), p99=("p99", "max"),
                       max=("max", "max"), saturated_frac=("saturated_frac", "max"))
                  .round(2))
display(per_excitation)
stats.to_csv(OUT_DIR / "band_statistics.csv", index=False)

**Why histograms on a log scale.** Intensities in a fluorescence cube span three orders of magnitude, and the interesting parts are at both ends: the noise floor on the left (its width tells you the read noise plus dark current) and the tail on the right (a spike at the ceiling is saturation, a long smooth tail is a bright fluorophore). On a linear scale both ends vanish into the axis. One curve per excitation, restricted to the mask, also shows immediately which excitations were exposed well: their histograms fill the range rather than piling up near zero.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for ex in data.excitation_wavelengths:
    v = data.get_excitation(ex).cube[mask].ravel()
    ax.hist(v[np.isfinite(v)], bins=120, histtype="step", log=True, label=f"ex {ex:.0f} nm")
ax.set(xlabel="intensity", ylabel="pixel-band count (log)", title="intensity distribution per excitation (inside the mask)")
ax.legend(fontsize=8, ncol=2)
plt.show()

**Why per-class spectra are the last statistic before modelling.** The papers evaluate band selection with a k-nearest-neighbour classifier trained on labelled regions; if the labelled classes do not differ spectrally at *some* excitation, no selection can help, so this plot is the go / no-go check for a dataset. It also shows which excitations separate which classes, a preview of what the selector should find.

Two annotation formats exist, for historical reasons. Lichens_2 uses the older one: a `.npy` class-id image (`-1` = unlabelled, `0..K-1` = classes) with a JSON legend (`lichens_2_class_info.json`, 0-indexed, with names, colours and pixel counts). The GUI wizard exports the newer one: `class_mask.png` (one colour per class, black background) plus `roi_regions.json` (1-indexed classes and rectangles). `spectral_select.load_ground_truth_from_png` reads the PNG form; both are documented in `docs/onboarding/DATA_GUIDE.md`. The cell uses whichever exists and skips cleanly otherwise.

In [ ]:
from spectral_select.viewer import compute_image_statistics

e = data.get_excitation(ex0)
img = e.cube[:, :, band_index(e, 520)]
print("whole image :", {k: round(float(v), 1) for k, v in compute_image_statistics(img).items()})
print("inside mask :", {k: round(float(v), 1) for k, v in compute_image_statistics(img, mask).items()})

# Per-class mean spectra from an annotation. Lichens_2 keeps class ids in a .npy (-1 = unlabelled) plus a JSON legend;
# the GUI exports the same information as class_mask.png + roi_regions.json (see DATA_GUIDE.md for both).
class_npy = PROC_SAMPLE_DIR / "lichens_2_class_mask.npy"
class_json = PROC_SAMPLE_DIR / "lichens_2_class_info.json"
if class_npy.exists() and np.load(class_npy).shape == data.spatial_shape:
    class_mask = np.load(class_npy)
    legend = json.loads(class_json.read_text()) if class_json.exists() else {}
    fig, axes = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [1, 1.6]})
    axes[0].imshow(np.ma.masked_less(class_mask, 0), cmap="tab10", interpolation="nearest")
    axes[0].set_title("class ids (-1 = unlabelled)")
    axes[0].axis("off")
    for cid in np.unique(class_mask[class_mask >= 0]):
        m, _ = roi_mean_spectrum(e, class_mask == cid)
        axes[1].plot(e.emission_wavelengths, m, label=legend.get(str(cid), {}).get("name", f"class {cid}"))
    axes[1].set(xlabel="emission (nm)", ylabel="mean intensity", title=f"class mean spectra at ex {ex0:.0f} nm")
    axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("no class annotation for this sample; see docs/onboarding/DATA_GUIDE.md for the annotation formats")

## 10. Preprocessing with the functional API

### Why pure functions, and why the GUI is built on them

The preprocessing wizard (`spectral-select-gui`) applies a fixed sequence: load, verify metadata, normalise, spatial crop, spectral crop, draw classes, ROI rectangles, export. When it was written (June 2026) each step was implemented as a **pure function** in `mehsi_preprocessor.processing` that takes a `SpectraData` and returns a new one, never mutating its input. The GUI buttons call those functions and nothing else. The reasons: the functions can be unit-tested without a display, a notebook or a script can reproduce exactly what a colleague clicked, and a bug fixed in one place is fixed everywhere. Below is the same pipeline as code.

### The science of each step

* **Rayleigh cutoff.** Excitation light scatters elastically off the sample and the optics and reaches the detector at the excitation wavelength. The tunable filter has a finite passband (about 10 to 20 nm, broader towards the red), so the scattered line leaks into the first bands above the excitation. Those bands are not fluorescence and are removed: everything below `ex + offset`.
* **Second-order cutoff.** The monochromator in the light source is a diffraction grating, and a grating set to pass `λ` also passes `2λ` (the second diffraction order). A residual line at twice the excitation therefore illuminates the sample and scatters into the detector at `2 ex`. The bands within `2 ex +/- offset` are removed. For the 310 nm excitation that is 580 to 660 nm, a big bite out of the visible range, which is why the 310 nm cube keeps only 22 bands.
* **Normalisation.** Divide by exposure and by lamp power (section 3) so that intensities are comparable across excitations. The GUI divides by the raw values; the older `HyperspectralProcessor` multiplies by a ratio to a reference (the longest exposure, the weakest lamp) so that the numbers stay in the familiar count range. Both are the same correction up to a global constant; do not mix outputs of the two in one model.
* **Spatial crop.** Restrict to the region of interest to save memory and training time. The mask is cropped with the cubes so they stay aligned.
* **Masked export.** `export_masked_pkl` writes `NaN` into every pixel outside the mask and stores the binary mask. Encoding "no data" *inside* the array means that any consumer, including the autoencoder's masked loss, cannot accidentally train on the background.

The cell starts from **uncut** cubes so that you can watch the cutoffs act. The loader dialect keeps them under `raw_data`; otherwise the loader from section 4a is reused; for synthetic data there is nothing to undo.

In [ ]:
from mehsi_preprocessor.processing.cropping import spatial_crop
from mehsi_preprocessor.processing.export import export_masked_pkl
from mehsi_preprocessor.processing.normalization import normalize_spectra
from mehsi_preprocessor.processing.spectral_filter import apply_rayleigh_cutoff


def uncut_from_loader_pickle(pkl: dict, meta: pd.DataFrame | None) -> SpectraData:
    exc = {}
    for ex_str, rec in pkl["raw_data"].items():
        ex = float(ex_str)
        row = meta[meta.excitation_nm == ex] if meta is not None else None
        exc[ex] = ExcitationData(excitation_nm=ex, cube=rec["data"], emission_wavelengths=list(rec["em_arr"]),
                                 exposure_time=rec.get("expos_val"), laser_power=_val(row, "power_W"))
    return SpectraData(excitations=exc, sample_name="uncut")


if "raw_data" in raw_pkl:
    uncut = uncut_from_loader_pickle(raw_pkl, meta)
elif loader is not None:
    uncut = SpectraData(
        excitations={ex: ExcitationData(ex, loader.raw_data[str(ex)]["data"], list(loader.raw_data[str(ex)]["em_arr"]),
                                        exposure_time=loader.raw_data[str(ex)]["expos_val"])
                     for ex in loader.excitation_wavelengths},
        sample_name="uncut")
else:
    uncut = data      # the synthetic data has no cutoff to undo

cut = apply_rayleigh_cutoff(uncut, cutoff_offset=40, apply_second_order=True)
normed = normalize_spectra(cut, by_exposure=True, by_laser_power=True)
H, W = normed.spatial_shape
cropped = spatial_crop(normed, roi=(H // 4, 3 * H // 4, W // 4, 3 * W // 4))

print(f"{'excitation':>10} {'uncut':>6} {'cut':>5}   removed emission bands")
for ex in uncut.excitation_wavelengths:
    a, b = uncut.get_excitation(ex), cut.get_excitation(ex)
    removed = sorted(set(a.emission_wavelengths) - set(b.emission_wavelengths))
    print(f"{ex:10.0f} {a.n_bands:6d} {b.n_bands:5d}   {removed[:3]}{' ...' if len(removed) > 3 else ''}")
print("cropped to", cropped.spatial_shape, "| normalised max at first excitation:",
      f"{float(np.nanmax(normed.get_excitation(ex0).cube)):.4g}")

masked_path = OUT_DIR / "spectra_masked.pkl"
export_masked_pkl(cut, mask.astype(np.uint8), masked_path)     # NaN outside the mask + binary mask, the GUI step 8 format
with open(masked_path, "rb") as fh:
    print("exported", masked_path.name, "->", list(pickle.load(fh).keys()))

**Why plot the cutoff on the mean spectrum.** The table above says *which* bands were removed; the plot says *whether that was the right call*. The grey curve is the mean spectrum inside the mask before the cutoffs, the red dots are the bands that survive, and the dashed lines mark `ex + 40`, `2 ex - 40` and `2 ex + 40`. For 365 nm the Rayleigh line at 405 nm is below the first band, so nothing is lost at the blue end, and the second-order window around 730 nm removes the last four bands where a scatter tail would otherwise sit. Repeat this plot for the 310 nm excitation and you will see the window in the middle of the visible range. If a real emission peak of your sample ever falls inside a window, that is the moment to argue for a smaller offset, with this figure as evidence.

In [ ]:
ex_show = 365.0 if 365.0 in uncut.excitations else uncut.excitation_wavelengths[0]
a, b = uncut.get_excitation(ex_show), cut.get_excitation(ex_show)
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(a.emission_wavelengths, np.nanmean(a.cube[mask], axis=0), "o-", color="0.6", label="uncut mean spectrum")
ax.plot(b.emission_wavelengths, np.nanmean(b.cube[mask], axis=0), "o", color="C3", label="kept after the cutoffs")
top = ax.get_ylim()[1]
for x, lab in [(ex_show + 40, "Rayleigh: ex + 40"), (2 * ex_show - 40, "2 ex - 40"), (2 * ex_show + 40, "2 ex + 40")]:
    ax.axvline(x, ls="--", color="k", alpha=0.4)
    ax.text(x, top * 0.95, lab, rotation=90, va="top", fontsize=8)
ax.set(xlabel="emission (nm)", ylabel="mean intensity", title=f"dual cutoff at excitation {ex_show:.0f} nm (offset 40 nm)")
ax.legend()
plt.show()

## 11. From data to the algorithm: a five-minute preview

### Why an autoencoder, and why perturbation

The goal of the project is to find the few (excitation, emission) bands that carry the information in a dataset, so that a cheaper instrument, or a downstream classifier, can keep only those. Three families of methods exist. *Marginal* scores (variance, PCA loadings) look at one band at a time and are fooled by redundancy: the brightest bands are all copies of each other. *Supervised* scores (mutual information with labels) work well but need labels, which in biomedical imaging are scarce and expensive. Our method is *unsupervised and joint*: a **masked convolutional autoencoder** is trained to reconstruct the whole 4D cube from a compact latent representation, so it must learn the joint spatial-spectral structure. We then **perturb** each important latent dimension and measure, band by band, how much the reconstruction reacts: bands the model relies on to explain the data react strongly, redundant ones do not. The scores are aggregated over dimensions, magnitudes and directions and normalised, and the top bands are selected with a diversity criterion so that the selection is not a cluster of neighbours. `Analyzer` wraps the whole thing; the algorithm itself lives in `selection_core` and is shared with the non-imaging work in `channel_select`.

### What the preview deliberately does not do

Below is a tiny run: a 64 x 64 crop around our ROI, three epochs, CPU, eight bands. It exists to show the mechanics and the output format in under a minute; the paper runs use the full image, one hundred to five hundred epochs and a GPU. Note the explicit `model_path`: without it, `Config` silently reuses `model_output/<sample_name>/model.pth` from a previous run, a trap that has cost people afternoons. The influence heatmap at the end is the picture the papers build on: excitation on one axis, emission on the other, score as colour, selected bands marked.

In [ ]:
from IPython.display import Image, display
from spectral_select import Analyzer, Config, Visualizer

if os.environ.get("HSI_WALKTHROUGH_SKIP_TRAIN"):
    print("skipping the training preview (HSI_WALKTHROUGH_SKIP_TRAIN is set)")
else:
    H, W = cut.spatial_shape
    r0, c0 = max(0, min(row - 32, H - 64)), max(0, min(col - 32, W - 64))
    r1, c1 = min(r0 + 64, H), min(c0 + 64, W)
    small = spatial_crop(cut, roi=(r0, r1, c0, c1))
    small_mask = mask[r0:r1, c0:c1]
    small = SpectraData(excitations=small.excitations, sample_name="walkthrough_preview",
                        mask=small_mask.astype(np.uint8) if small_mask.any() else None)

    config = Config(
        sample_name="walkthrough_preview",
        n_bands_to_select=8,
        training_epochs=3,                          # toy setting; the paper runs use 100 to 500
        device="cpu",
        model_path=OUT_DIR / "preview_model.pth",   # explicit, otherwise Config silently reuses model_output/<sample>/model.pth
        output_dir=OUT_DIR / "preview_results",
        save_tiff_layers=False,
    )
    t0 = time.time()
    analyzer = Analyzer(config).fit(small)
    print(f"\ntrained + selected in {time.time() - t0:.0f}s")
    for band in analyzer.get_wavelengths():
        print(f"  rank {band.rank:2d}  ex {band.excitation_nm:5.0f} nm  em {band.emission_nm:5.0f} nm  "
              f"influence {band.influence_score:.3e}")

    viz = Visualizer.from_analyzer(analyzer, output_dir=OUT_DIR / "preview_results")
    heatmap_png = viz.plot_influence_heatmap()
    display(Image(filename=str(heatmap_png), width=700))

### SpectraForge: synthetic data with known answers

**Why we built a simulator.** Real data has no ground truth for "which bands matter": you can score a selection by how well a classifier does with it, but you cannot say which bands *should* have been chosen. SpectraForge answers that by generating data whose answer is known. You define fluorophores (Gaussian excitation and emission spectra with literature peak positions), mix them into materials, paint the materials onto a scene, and describe the instrument. The forward model mirrors what section 1 described: the signal at a pixel is the sum over fluorophores of concentration x extinction x quantum yield x excitation efficiency at `λex` x emission profile at `λem`, scaled by lamp, exposure and power, plus a Rayleigh line at the excitation, a second-order line at twice the excitation, Poisson photon noise and Gaussian read noise. The `GroundTruth` sidecar records each fluorophore's concentration map and the clean, noise-free cube, from which `informative_bands()` says which emission bands carry signal at each excitation.

**A note of scientific honesty.** An early SpectraForge result claiming that the selector recovers realistic spectra was withdrawn in July 2026: the ground-truth mask was so broad that a random selector scored the same. The harness now ships a random baseline and a tight peak-recovery metric, and the current verdict is "inconclusive". Read `docs/onboarding/components/01-spectraforge.md` before quoting any synthetic number. The output still loads exactly like real data, which makes it the right sandbox for testing code paths without a microscope.

In [ ]:
from spectraforge import AcquisitionConfig, ArtifactConfig, Material, Scene, load_builtin_library, render

lib = load_builtin_library()                    # collagen, NADH, FAD, EGFP, ... (parametric Gaussian spectra)
scene = Scene(48, 48)
scene.paint_rect(Material("tissue", {"collagen": 1.0, "NADH": 0.4}), 4, 30, 4, 30)
scene.paint_circle(Material("fad_spot", {"FAD": 1.0}), cy=34, cx=34, radius=9)
acq = AcquisitionConfig(excitations=[340.0, 365.0, 450.0], em_min=400, em_max=700, em_step=10)
synthetic, gt = render(scene, lib, acq, artifacts=ArtifactConfig(rayleigh_strength=0.1, photon_scale=300.0), seed=1)

print("synthetic:", synthetic.n_excitations, "excitations,", synthetic.spatial_shape, "| loads in the GUI / Analyzer like real data")
grid = np.asarray(gt.emission_grid)
for ex, informative in gt.informative_bands(threshold=0.05).items():
    wl = grid[informative]
    print(f"  ex {ex:.0f} nm: {int(informative.sum()):2d} bands carry signal ({wl.min():.0f}-{wl.max():.0f} nm)")
synthetic.to_pickle(OUT_DIR / "synthetic_spectra.pkl")
gt.save(OUT_DIR)
print("wrote synthetic_spectra.pkl + groundtruth.npz/json to", OUT_DIR)

## 12. Where next

* **Run the GUI wizard on the same folder**: `spectral-select-gui` (ten steps, load to select). The pure functions used in section 10 are exactly what its buttons call.
* **Desktop viewer** for quick looks without Jupyter: `python -c "from spectral_select import launch_viewer; launch_viewer()"`.
* **Draw regions inside a notebook**: `from spectral_select import ROIWidget` (needs `%matplotlib widget` and `ipywidgets`), then `widget.to_ground_truth()`.
* `examples/01_quickstart.ipynb` and `examples/02_validation.ipynb` continue with a full selection run and ground-truth validation.
* Component guides live in `docs/onboarding/components/` (SpectraForge, the preprocessor, spectral_select, selection_core, channel_select, experiments).
* Dataset contract and catalogue: `docs/onboarding/DATA_GUIDE.md`.

Scratch outputs from this session are in the directory printed at the top. Delete it whenever you like.